DADOS COMPLETOS

# Pipeline de experimentos PLN

Usando subconjunto dos nossos dados (rotulação ainda provisória).

Estou montando meu Google Drive, pois estou lendo o arquivo de entrada de lá.

In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


## 1. Estruturação dos dados

Primeiro, vamos importar o conjunto de dados.

In [4]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

import pandas as pd
dados = pd.read_csv('./drive/My Drive/Colab Notebooks/Posicionamento/conjuntoDeDados.tsv', sep='\t', decimal = ',', encoding = 'UTF-8')
dados

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,target_id,target_message,id,parent_id,author,parent_name,parent_message,message,parent_label,target_parent_message,target_and_message,label
0,1,Eu acho que a direita está sendo hipócrita ao ...,jun76kr,15h8cpw,Gerald_Huber,Raiz,NaN,"Incoerência da direita: pregam menos Estado, l...",Alvo da conversa,Eu acho que a direita está sendo hipócrita ao ...,Eu acho que a direita está sendo hipócrita ao ...,Concorda
1,1,Eu acho que a direita está sendo hipócrita ao ...,juna2pp,jun76kr,Jeremy_Newman,Gerald_Huber,"Incoerência da direita: pregam menos Estado, l...","E olha que aqui tá cheio de liberais no sub, m...",Concorda,Eu acho que a direita está sendo hipócrita ao ...,Eu acho que a direita está sendo hipócrita ao ...,Concorda
2,1,Eu acho que a direita está sendo hipócrita ao ...,junarew,juna2pp,Gerald_Huber,Jeremy_Newman,"E olha que aqui tá cheio de liberais no sub, m...",Daí dizem que defender o controle do Estado na...,Concorda,Eu acho que a direita está sendo hipócrita ao ...,Eu acho que a direita está sendo hipócrita ao ...,Concorda
3,1,Eu acho que a direita está sendo hipócrita ao ...,junbgq9,junarew,Jeremy_Newman,Gerald_Huber,Daí dizem que defender o controle do Estado na...,Faz uma enquete sobre legalização do aborto pa...,Concorda,Eu acho que a direita está sendo hipócrita ao ...,Eu acho que a direita está sendo hipócrita ao ...,Concorda
4,1,Eu acho que a direita está sendo hipócrita ao ...,juncvrw,junbgq9,Gerald_Huber,Jeremy_Newman,Faz uma enquete sobre legalização do aborto pa...,Pois é... outra vez: aborto não é questão de E...,Concorda,Eu acho que a direita está sendo hipócrita ao ...,Eu acho que a direita está sendo hipócrita ao ...,Concorda
...,...,...,...,...,...,...,...,...,...,...,...,...
2261,249,Com toda essa discussão sobre as regras do pix...,m6n8401,m6n3kc8,Anthony_Griffin,Robert_Wong,O correto seria o governo também criar um cana...,Mano vc esqueceu o /s,Discorda,Com toda essa discussão sobre as regras do pix...,Com toda essa discussão sobre as regras do pix...,Discorda
2262,249,Com toda essa discussão sobre as regras do pix...,m6n8r12,m6n3kc8,Sheri_Fry,Robert_Wong,O correto seria o governo também criar um cana...,FUCKING AMBULANTES e comércios menores vão ter...,Discorda,Com toda essa discussão sobre as regras do pix...,Com toda essa discussão sobre as regras do pix...,Discorda
2263,249,Com toda essa discussão sobre as regras do pix...,m6n6kpy,m6n3kc8,Christopher_Shaw,Robert_Wong,O correto seria o governo também criar um cana...,Só se mudar a legislação. Eles não são obrigad...,Discorda,Com toda essa discussão sobre as regras do pix...,Com toda essa discussão sobre as regras do pix...,Discorda
2264,249,Com toda essa discussão sobre as regras do pix...,m6oa15g,m6n3kc8,Sheila_Bishop,Robert_Wong,O correto seria o governo também criar um cana...,[removed],Discorda,Com toda essa discussão sobre as regras do pix...,Com toda essa discussão sobre as regras do pix...,Irrelevante


Agora, vamos definir as variáveis de interesse (ou seja, a variável texto e a categoria que será prevista). Além disso, vamos utilizar o *Counter* para avaliar se há desbalanceamento entre as classes.

In [5]:
from math import nan
from collections import Counter

dados = dados[(dados['label'] != 'Alvo da conversa') & (dados['label'] != 'Comentário Original') & (dados['label'].notna())]


dados['label'] = dados['label'].replace('Discute', 'Outros')
dados['label'] = dados['label'].replace('Irrelevante', 'Outros')
dados['label'] = dados['label'].replace('Pede Informações', 'Outros')

dados['parent_label'] = dados['parent_label'].replace('Discute', 'Outros')
dados['parent_label'] = dados['parent_label'].replace('Irrelevante', 'Outros')
dados['parent_label'] = dados['parent_label'].replace('Pede Informações', 'Outros')

X = dados['target_and_message']
y = dados['label']
#y = dados['target_id']

Counter(y)

Counter({'Concorda': 942, 'Outros': 702, 'Discorda': 622})

Por fim, vamos dividir o conjunto de dados em duas partições: a de treinamento e a de teste. Isso será feito utilizando o comando *stratify* para que sejam mantidas as mesmas proporções entre as classes no conjunto de treinamento e de teste.

In [6]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.19947, shuffle=False)

Counter(y_test)

Counter({'Concorda': 161, 'Discorda': 101, 'Outros': 190})

## 2. Criação de modelos

Agora, vamos criar os modelos que serão testados na última etapa. Primeiro, vamos importar as bibliotecas que utilizaremos:

In [7]:
# Pipelines
from sklearn.pipeline import Pipeline

# Engenharia de Características (features)
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import MinMaxScaler

# Modelos
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import MultinomialNB

from sklearn import svm

# Métricas
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, confusion_matrix

Não foi utilizado Grid Search ainda (para otimização de hiperparâmetros).

In [8]:
from unicodedata import normalize

minhaStopList = [normalize('NFKD', n).encode('ASCII', 'ignore').decode('ASCII') for n in stopwords.words('portuguese')]
#minhaStopList.append('ii')


In [9]:
baseline = DummyClassifier(strategy='most_frequent', random_state = 100, constant = None)
reglog = LogisticRegression(class_weight='balanced', max_iter=2000, solver='liblinear', penalty='l2', C=1.3225)
mlp = MLPClassifier(activation='relu', max_iter=2000)
NB = MultinomialNB()

SVM = svm.SVC(kernel='poly', C = 1, gamma=1)

## 3. Comparação dos modelos

Agora, vamos criar um pipeline de pipelines para comparar diferentes combinações de modelos e pré-processamentos diferentes. Abaixo, crio uma lista com todas as combinações que quero testar.

OBS.: É importante dar um nome bom para cada uma das etapas; isso vai ajudar a identificar o modelo em questão na fase de comparações.

In [10]:
# modelos1: conjunto de modelos que não usam seleção de atributos
modelos1 = [Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('dummy', baseline)]),
           Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('NB', NB)]),
           Pipeline(steps=[('tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,)),
                              ('NB', NB)]),
           Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('relog', reglog)]),
           Pipeline(steps=[('tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,)),
                              ('relog', reglog)]),
           Pipeline(steps=[('bigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('NB', NB)]),
           Pipeline(steps=[('trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('NB', NB)]),
           Pipeline(steps=[('uni-bi-trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('NB', NB)]),
           Pipeline(steps=[('bigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('NB', NB)]),
           Pipeline(steps=[('trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('NB', NB)]),
           Pipeline(steps=[('uni-bi-trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('NB', NB)]),
           Pipeline(steps=[('bigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('relog', reglog)]),
           Pipeline(steps=[('trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('relog', reglog)]),
           Pipeline(steps=[('uni-bi-trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('relog', reglog)]),
           Pipeline(steps=[('bigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('relog', reglog)]),
           Pipeline(steps=[('trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('relog', reglog)]),
           Pipeline(steps=[('uni-bi-trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('relog', reglog)]),
           Pipeline(steps=[('bigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('mlp', mlp)]),
           Pipeline(steps=[('trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('mlp', mlp)]),
           Pipeline(steps=[('uni-bi-trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('mlp', mlp)]),
           Pipeline(steps=[('bigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('mlp', mlp)]),
           Pipeline(steps=[('trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('mlp', mlp)]),
           Pipeline(steps=[('uni-bi-trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('mlp', mlp)]),
           Pipeline(steps=[('bigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('SVM', SVM)]),
           Pipeline(steps=[('trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('SVM', SVM)]),
           Pipeline(steps=[('uni-bi-trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('SVM', SVM)]),
           Pipeline(steps=[('bigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('SVM', SVM)]),
           Pipeline(steps=[('trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('SVM', SVM)]),
           Pipeline(steps=[('uni-bi-trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('SVM', SVM)]),
           ]

# modelos2: conjunto de modelos que usam seleção de atributos com a função SelectKBest
atributos  = 300
modelos2 = [Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('dummy', baseline)]),
           Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,)),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('NB', NB)]),
           Pipeline(steps=[('tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,)),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('NB', NB)]),
           Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,)),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('reglog', reglog)]),
           Pipeline(steps=[('tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,)),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('reglog', reglog)]),
           Pipeline(steps=[('bigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('NB', NB)]),
           Pipeline(steps=[('trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('NB', NB)]),
           Pipeline(steps=[('uni-bi-trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('NB', NB)]),
           Pipeline(steps=[('bigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('NB', NB)]),
           Pipeline(steps=[('trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('NB', NB)]),
           Pipeline(steps=[('uni-bi-trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('NB', NB)]),
           Pipeline(steps=[('bigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('relog', reglog)]),
           Pipeline(steps=[('trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('relog', reglog)]),
           Pipeline(steps=[('uni-bi-trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('relog', reglog)]),
           Pipeline(steps=[('bigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('relog', reglog)]),
           Pipeline(steps=[('trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('relog', reglog)]),
           Pipeline(steps=[('uni-bi-trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('relog', reglog)]),
           Pipeline(steps=[('bigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('uni-bi-trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('bigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('uni-bi-trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('bigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('uni-bi-trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('bigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('uni-bi-trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('SVM', SVM)]),
           ]

# modelos3: conjunto de modelos que utilizam redução de dimensionalidade usando Análise de Componentes Principais (PCA)
modelos3 = [Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('pca', TruncatedSVD(atributos)),
                              ('dummy', baseline)]),
           Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('pca', TruncatedSVD(atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,)),
                              ('pca', TruncatedSVD(atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('pca', TruncatedSVD(atributos)), ('Normalizing',MinMaxScaler()),
                              ('NB', NB)]),
           Pipeline(steps=[('tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,)),
                              ('pca', TruncatedSVD(atributos)), ('Normalizing',MinMaxScaler()),
                              ('NB', NB)]),
           Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('pca', TruncatedSVD(atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,)),
                              ('pca', TruncatedSVD(atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('pca', TruncatedSVD(atributos)),
                              ('reglog', reglog)]),
           Pipeline(steps=[('tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,)),
                              ('pca', TruncatedSVD(atributos)),
                              ('reglog', reglog)]),
           Pipeline(steps=[('bigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('pca', TruncatedSVD(atributos)), ('Normalizing',MinMaxScaler()),
                              ('NB', NB)]),
           Pipeline(steps=[('trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('pca', TruncatedSVD(atributos)), ('Normalizing',MinMaxScaler()),
                              ('NB', NB)]),
           Pipeline(steps=[('uni-bi-trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('pca', TruncatedSVD(atributos)), ('Normalizing',MinMaxScaler()),
                              ('NB', NB)]),
           Pipeline(steps=[('bigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('pca', TruncatedSVD(atributos)), ('Normalizing',MinMaxScaler()),
                              ('NB', NB)]),
           Pipeline(steps=[('trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('pca', TruncatedSVD(atributos)), ('Normalizing',MinMaxScaler()),
                              ('NB', NB)]),
           Pipeline(steps=[('uni-bi-trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('pca', TruncatedSVD(atributos)), ('Normalizing',MinMaxScaler()),
                              ('NB', NB)]),
           Pipeline(steps=[('bigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('pca', TruncatedSVD(atributos)),
                              ('relog', reglog)]),
           Pipeline(steps=[('trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('pca', TruncatedSVD(atributos)),
                              ('relog', reglog)]),
           Pipeline(steps=[('uni-bi-trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('pca', TruncatedSVD(atributos)),
                              ('relog', reglog)]),
           Pipeline(steps=[('bigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('pca', TruncatedSVD(atributos)),
                              ('relog', reglog)]),
           Pipeline(steps=[('trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('pca', TruncatedSVD(atributos)),
                              ('relog', reglog)]),
           Pipeline(steps=[('uni-bi-trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('pca', TruncatedSVD(atributos)),
                              ('relog', reglog)]),
           Pipeline(steps=[('bigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('pca', TruncatedSVD(atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('pca', TruncatedSVD(atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('uni-bi-trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('pca', TruncatedSVD(atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('bigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('pca', TruncatedSVD(atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('pca', TruncatedSVD(atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('uni-bi-trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('pca', TruncatedSVD(atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('bigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('pca', TruncatedSVD(atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('pca', TruncatedSVD(atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('uni-bi-trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('pca', TruncatedSVD(atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('bigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('pca', TruncatedSVD(atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('pca', TruncatedSVD(atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('uni-bi-trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('pca', TruncatedSVD(atributos)),
                              ('SVM', SVM)]),
           ]

# Cada bloco a seguir avalia um conjunto de modelos
Os modelos são avaliados considerando validação cruzada utilizando o conjunto de treinamento.

# Modelos que não utilizam nenhuma abordagem de seleção de atributos/redução de dimensionalidade.

# Modelos que utilizam seleção de atributos (utilizando a função SelectKBest).

In [17]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

kf = StratifiedKFold(n_splits=3, random_state = 123, shuffle = True)

testados = modelos2

for m in testados:
  print([i for i,j in m.steps], '\tf1_macro\t', round(cross_val_score(m, x_train, y_train, cv=kf, scoring='f1_macro').mean(), 3))
  print([i for i,j in m.steps], '\taccuracy\t', round(cross_val_score(m, x_train, y_train, cv=kf, scoring='accuracy').mean(), 3))
  #print([i for i,j in m.steps], '\troc_auc\t', round(cross_val_score(m, x_train, y_train, cv=kf, scoring='roc_auc').mean(), 3))

['bag-of-words', 'KBest', 'dummy'] 	f1_macro	 0.201
['bag-of-words', 'KBest', 'dummy'] 	accuracy	 0.431
['bag-of-words', 'KBest', 'mlp'] 	f1_macro	 0.45
['bag-of-words', 'KBest', 'mlp'] 	accuracy	 0.466
['tf-idf', 'KBest', 'mlp'] 	f1_macro	 0.445
['tf-idf', 'KBest', 'mlp'] 	accuracy	 0.474
['bag-of-words', 'KBest', 'NB'] 	f1_macro	 0.437
['bag-of-words', 'KBest', 'NB'] 	accuracy	 0.459
['tf-idf', 'KBest', 'NB'] 	f1_macro	 0.334
['tf-idf', 'KBest', 'NB'] 	accuracy	 0.463
['bag-of-words', 'KBest', 'SVM'] 	f1_macro	 0.425
['bag-of-words', 'KBest', 'SVM'] 	accuracy	 0.433
['tf-idf', 'KBest', 'SVM'] 	f1_macro	 0.304
['tf-idf', 'KBest', 'SVM'] 	accuracy	 0.471
['bag-of-words', 'KBest', 'reglog'] 	f1_macro	 0.456
['bag-of-words', 'KBest', 'reglog'] 	accuracy	 0.463
['tf-idf', 'KBest', 'reglog'] 	f1_macro	 0.446
['tf-idf', 'KBest', 'reglog'] 	accuracy	 0.478
['bigram', 'KBest', 'NB'] 	f1_macro	 0.408
['bigram', 'KBest', 'NB'] 	accuracy	 0.468
['trigram', 'KBest', 'NB'] 	f1_macro	 0.36
['trigra

In [16]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

kf = StratifiedKFold(n_splits=3, random_state = 123, shuffle = True)

testados = modelos1

for m in testados:
  print([i for i,j in m.steps], '\tf1_macro\t', round(cross_val_score(m, x_train, y_train, cv=kf, scoring='f1_macro').mean(), 3))
  print([i for i,j in m.steps], '\taccuracy\t', round(cross_val_score(m, x_train, y_train, cv=kf, scoring='accuracy').mean(), 3))
  #print([i for i,j in m.steps], '\troc_auc\t', round(cross_val_score(m, x_train, y_train, cv=kf, scoring='roc_auc').mean(), 3))

['bag-of-words', 'dummy'] 	f1_macro	 0.201
['bag-of-words', 'dummy'] 	accuracy	 0.431
['bag-of-words', 'mlp'] 	f1_macro	 0.437
['bag-of-words', 'mlp'] 	accuracy	 0.448
['tf-idf', 'mlp'] 	f1_macro	 0.432
['tf-idf', 'mlp'] 	accuracy	 0.437
['bag-of-words', 'NB'] 	f1_macro	 0.46
['bag-of-words', 'NB'] 	accuracy	 0.471
['tf-idf', 'NB'] 	f1_macro	 0.372
['tf-idf', 'NB'] 	accuracy	 0.482
['bag-of-words', 'SVM'] 	f1_macro	 0.406
['bag-of-words', 'SVM'] 	accuracy	 0.424
['tf-idf', 'SVM'] 	f1_macro	 0.396
['tf-idf', 'SVM'] 	accuracy	 0.469
['bag-of-words', 'relog'] 	f1_macro	 0.45
['bag-of-words', 'relog'] 	accuracy	 0.458
['tf-idf', 'relog'] 	f1_macro	 0.467
['tf-idf', 'relog'] 	accuracy	 0.477
['bigram', 'NB'] 	f1_macro	 0.434
['bigram', 'NB'] 	accuracy	 0.438
['trigram', 'NB'] 	f1_macro	 0.438
['trigram', 'NB'] 	accuracy	 0.443
['uni-bi-trigram', 'NB'] 	f1_macro	 0.441
['uni-bi-trigram', 'NB'] 	accuracy	 0.448
['bigram tf-idf', 'NB'] 	f1_macro	 0.414
['bigram tf-idf', 'NB'] 	accuracy	 0.452


# Modelos que utilizam redução de dimensionalidade usando Análise de Componentes Principais (PCA)

In [15]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

kf = StratifiedKFold(n_splits=3, random_state = 123, shuffle = True)

testados = modelos3

for m in testados:
  print([i for i,j in m.steps], '\tf1_macro\t', round(cross_val_score(m, x_train, y_train, cv=kf, scoring='f1_macro').mean(), 3))
  print([i for i,j in m.steps], '\taccuracy\t', round(cross_val_score(m, x_train, y_train, cv=kf, scoring='accuracy').mean(), 3))
  #print([i for i,j in m.steps], '\troc_auc\t', round(cross_val_score(m, x_train, y_train, cv=kf, scoring='roc_auc').mean(), 3))

['bag-of-words', 'pca', 'dummy'] 	f1_macro	 0.201
['bag-of-words', 'pca', 'dummy'] 	accuracy	 0.431
['bag-of-words', 'pca', 'mlp'] 	f1_macro	 0.457
['bag-of-words', 'pca', 'mlp'] 	accuracy	 0.46
['tf-idf', 'pca', 'mlp'] 	f1_macro	 0.458
['tf-idf', 'pca', 'mlp'] 	accuracy	 0.466
['bag-of-words', 'pca', 'Normalizing', 'NB'] 	f1_macro	 0.201
['bag-of-words', 'pca', 'Normalizing', 'NB'] 	accuracy	 0.431
['tf-idf', 'pca', 'Normalizing', 'NB'] 	f1_macro	 0.201
['tf-idf', 'pca', 'Normalizing', 'NB'] 	accuracy	 0.431
['bag-of-words', 'pca', 'SVM'] 	f1_macro	 0.412
['bag-of-words', 'pca', 'SVM'] 	accuracy	 0.426
['tf-idf', 'pca', 'SVM'] 	f1_macro	 0.358
['tf-idf', 'pca', 'SVM'] 	accuracy	 0.472
['bag-of-words', 'pca', 'reglog'] 	f1_macro	 0.455
['bag-of-words', 'pca', 'reglog'] 	accuracy	 0.463
['tf-idf', 'pca', 'reglog'] 	f1_macro	 0.467
['tf-idf', 'pca', 'reglog'] 	accuracy	 0.472
['bigram', 'pca', 'Normalizing', 'NB'] 	f1_macro	 0.23
['bigram', 'pca', 'Normalizing', 'NB'] 	accuracy	 0.437
['

Os diferentes modelos foram avaliados com validação cruzada (usando apenas o conjunto de treinamento). O melhor "deve" ser o selecionado e seu desempenho deverá ser avaliado com o conjunto de teste.

## 4. Teste do modelo escolhido

Testa-se o modelo escolhido (teoricamente o que obteve o melhor desempenho). O modelo é treinado com todo o conjunto de treinamento e testado com o conjunto de teste.

In [18]:
from sklearn import metrics # Métricas para avaliação da classificação


#melhorModelo = Pipeline(steps=[('trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),('pca', TruncatedSVD(atributos)), ('Normalizing',MinMaxScaler()),('NB', NB)])
#melhorModelo = modelos1[3];
melhorModelo = Pipeline(steps=[('uni-bi-trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))), ('pca', TruncatedSVD(atributos)), ('relog', reglog)])
melhorModelo.fit(x_train, y_train)

preds = melhorModelo.predict(x_test) ### escolha um modelo para verificar seu desempenho no conjunto de teste
report=metrics.classification_report(y_test,preds)
print(report)

# 1. (Opcional) Salvar predições e valores reais em um DataFrame
df_preds = pd.DataFrame({
    'y_real': y_test,
    'y_predito': preds
})

# 2. Caminho para salvar no seu Google Drive
caminho_arquivo_preds = './drive/My Drive/Colab Notebooks/Posicionamento/predicoesConjuntoDeDados.csv'
caminho_arquivo_relatorio = './drive/My Drive/Colab Notebooks/Posicionamento/relatorio_classificacaoConjuntoDeDados.txt'

# 3. Salvar predições como CSV
df_preds.to_csv(caminho_arquivo_preds, index=False)

# 4. Salvar relatório como TXT
with open(caminho_arquivo_relatorio, 'w') as f:
    f.write(report)

print("Arquivos salvos no Google Drive com sucesso.")

              precision    recall  f1-score   support

    Concorda       0.38      0.63      0.48       161
    Discorda       0.37      0.11      0.17       101
      Outros       0.55      0.46      0.50       190

    accuracy                           0.44       452
   macro avg       0.44      0.40      0.38       452
weighted avg       0.45      0.44      0.42       452

Arquivos salvos no Google Drive com sucesso.


Pipeline de experimentos adaptado de notebook desenvolvido pela ex-aluna Laís Carraro.